# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules (these will be checked in the model answer review next week)
1. Every bar chart **must have a zero baseline** — no exceptions (SWD p.51)
2. Every chart **must have an insight title**, not a topic title (SWD p.29)
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names (SWD p.57)

---


## Setup — Run this cell first


In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/content/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [3]:
!pip install plotly==6.1.1 kaleido==0.2.1

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [4]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean().reset_index().sort_values('Happiness_Score'))

# Step 2: Assign unique Vivid colour to each region
bar_colors = px.colors.qualitative.Vivid[:len(region_avg)]

# Step 3: Build horizontal bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    x=region_avg['Happiness_Score'], y=region_avg['Region'],
    orientation='h', marker_color=bar_colors, marker_opacity=0.9,
    text=region_avg['Happiness_Score'].round(2), textposition='outside'
))

# Step 4: Clean up layout
fig.update_layout(
    title='Western Europe leads global happiness by a wide margin<br><sup>Scores reflect wealth, freedom and social support</sup>',
    xaxis=dict(range=[0, 9], gridcolor='#EEEEEE', showgrid=True),
    plot_bgcolor='#FAFAFA', bargap=0.25, height=500
)

fig.show()
fig.write_image('task1.png')



---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [7]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
# Step 1: Separate top 8 and bottom 8 countries and label each group
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

# Step 2: Combine both groups into one dataframe
combined = pd.concat([bottom8, top8])
global_avg = df['Happiness_Score'].mean()

# Step 3: Build horizontal bar chart with colour per group
fig = px.bar(
    combined, x='Happiness_Score', y='Country',
    orientation='h', color='Group',
    color_discrete_map={'Top 8': px.colors.qualitative.Vivid[0],
                        'Bottom 8': px.colors.qualitative.Vivid[1]},
    text=combined['Happiness_Score'].round(2),
    title='The happiest and least happy nations are 4+ points apart<br><sup>A gap that reflects inequality in wealth, freedom and social support</sup>'
)

# Step 4: Style the bars and add global average reference line
fig.update_traces(textposition='outside', marker_opacity=0.9)
fig.add_vline(x=global_avg, line_dash='dash', line_color='#333333',
              annotation_text=f'Global avg {global_avg:.2f}')

# Step 5: Clean up layout
fig.update_layout(
    xaxis=dict(range=[0, 9.5], gridcolor='#EEEEEE', showgrid=True),
    plot_bgcolor='#FAFAFA', bargap=0.15, height=550
)

fig.show()
fig.write_image('task2.png')


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [8]:
# Stretch goal — grouped bar chart
# YOUR CODE HERE
# Step 1: Filter to 5 key regions
target_regions = ['Western Europe', 'Latin America and Caribbean',
                  'East Asia', 'Sub-Saharan Africa', 'South Asia']

# Step 2: Compute average GDP and Freedom per region
stretch_df = (df[df['Region'].isin(target_regions)]
              .groupby('Region')[['GDP', 'Freedom']]
              .mean().reset_index().sort_values('GDP', ascending=False))

# Step 3: Melt to long form for grouped bars
stretch_long = stretch_df.melt(id_vars='Region', value_vars=['GDP', 'Freedom'],
                                var_name='Factor', value_name='Score')

# Step 4: Build grouped bar chart
fig = px.bar(
    stretch_long, x='Region', y='Score', color='Factor',
    barmode='group', color_discrete_sequence=px.colors.qualitative.Vivid,
    text=stretch_long['Score'].round(2),
    title='Freedom is more evenly spread than wealth<br><sup>Western Europe leads on GDP but the freedom gap is far smaller</sup>'
)

# Step 5: Style the bars and clean up layout
fig.update_traces(textposition='outside', marker_opacity=0.9)
fig.update_layout(
    yaxis=dict(range=[0, 1.25], showgrid=True, gridcolor='#EEEEEE'),
    plot_bgcolor='#FAFAFA', bargap=0.25, height=500
)

fig.show()
fig.write_image('task3.png')
